# How Meaning is Represented in Computers? Tokenization & Embeddings 

## Background

In the last four computational essays, we codified the attention mechanism, starting from the basic self-attention mechanism and progressing all the way to trainable weight-based multi-head causal masked attention meachanism. Now, in this computational essay, let's try to understand how meaning is represented in computer systems and how modern LLMs practically achieve the remarkable ability to understand human intent expressed in their own language in a mysterious way. But after experiencing this computational essay, it will no longer remain a mystery to you.

## Why

Text has a fundamental relationship with meaning because [God Himself taught human a capability](https://quran.com/al-baqarah/31?readingMode=verse-by-verse&translations=84) that no other creature has been given: the ability to acquire knowledge through words.

On a deeper level: the ability to see the singular universal term in the plurality of particular things.

Knowledge is not words.

Words are merely representations of knowledge.

A word is a vessel through which abstract meaning is transferred to other human beings, allowing us to represent plurality with a single word.

In short, the essence of a word is not the word itself, but the meaning behind it.

Computers have been storing and manipulating textual data for decades, but they were never as good at reasoning and understanding it as they are now with the advent of LLMs.

Traditionally, computers represented words in memory using encodings such as ASCII and manipulated or sorted them as discrete symbols. However, this discrete representation of words is not sufficient for reasoning. To reason in a way that resembles human language understanding, computers need a way to represent the field of meanings conveyed by words.

The field of meanings of a word is a continuous concept that can be represented as a vector of *n* dimensions. The greater the number of dimensions, the richer the representation of the field of meanings conveyed by the word, at the cost of increased computational power.

For example, consider the word **"King."** We choose a set of properties and assign continuous values to each of them. Together, these values form a vector of *n* dimensions, where *n* is the number of properties we choose to represent the meaning of the word in a computer system.

### Example properties

* **authority**: a continuous numerical value representing the level of authority.
* **gender**: a binary value representing the concept of masculinity or femininity.
* **event**: a continuous value representing whether the word refers to an event or a person.
* **rich**: a continuous numerical value representing the level of wealth.
* **wings**: a continuous numerical value representing the concept of having wings.

Let's consider a small vocabulary and encode the meanings of these words based on the above properties.

| Properties | "King" | "Eagle" | "Battle" | "Man" | "Woman" | "Queen" |
| ---------- | -----: | ------: | -------: | ----: | ------: | ------: |
| authority  |     10 |       1 |        0 |    10 |      10 |      10 |
| gender     |      1 |       1 |        0 |     1 |       0 |       0 |
| event      |      0 |       0 |       10 |     0 |       0 |       0 |
| rich       |     10 |     0.1 |      0.1 |     1 |     0.9 |      10 |
| wings      |      0 |      10 |        0 |     0 |       0 |       0 |

These properties can be represented as vectors:

* **"King"** = [10, 1, 0, 10, 0]
* **"Eagle"** = [1, 1, 0, 0.1, 10]
* **"Battle"** = [0, 0, 10, 0.1, 0]
* **"Man"** = [10, 1, 0, 1, 0]
* **"Woman"** = [10, 0, 0, 0.9, 0]
* **"Queen"** = [10, 0, 0, 10, 0]

These representations give us a way to work with the field of meanings behind words rather than with the words themselves.

For example, the following arithmetic on meanings becomes possible:

**"King" − "Man" + "Woman" = "Queen"**

`=> [10, 1, 0, 10, 0] − [10, 1, 0, 1, 0] + [10, 0, 0, 0.9, 0] = [10, 0, 0, 9.9, 0]`

The resulting vector, **[10, 0, 0, 9.9, 0]**, is very close to the vector representing **"Queen"**, **[10, 0, 0, 10, 0]**.

This is the core idea behind representing textual data together with its field of meaning in modern LLMs. It enables them to understand the intent expressed by humans through language in a way that resembles human language understanding.

However, remember the limitations: 
- _We must choose a fixed number of dimensions to represent meaning. This constraint is one of the reasons LLMs will always remain behind humans._
- _LLMs merely attempt to represent the field of meanings in which humans have used words so far up to this point in timeline. However, the creativity to use the same words to convey entirely new dimensions of abstract meaning i.e. enrich existing meanings or to signify new meanings altogether—still remains uniquely human._

And this is the core essence of being human.

## Goal of this Computational Essay

Given a large textual dataset, break it down into words and subwords, known technically as **tokens**, represent them as vectors, known technically as **embeddings**, and prepare them to be fed into the GPT model (with the attention mechanism we developed in the previous computational essay) for training.

<img src="images/text-embedding.png" width="50%" />

<br />

## Recipe

1. Read a small text file simulated as an input corpus for LLM training in this computational essay.
2. Build the vocabulary by extracting unique words from the corpus. 
3. Split the text into words and subwords, known technically as **tokens**, using:
  - (a). Simple tokenization approach,
  - (b). Enhanced tokenization approach, and
  - (c). Production-grade tokenization algorithm used in modern LLMs.
4. Generate training samples from the input corpus using a sliding window for self-supervised learning.
  - (a). Essence of the data sampling,
  - (b). Production-grade data sampling approach.
5. Map each token ID to a dense vector representation, known as an **embedding** technically.
6. Make the token embeddings aware of the position where they are used in the sequence.
7. Stitch together.

<img src="images/embedding-pipeline-overview.png" width="50%" />


## Step 1: Load Input Text Data as a Training Corpus

For demonstration purposes, we use a relatively small text file to understand how textual data is prepared for LLM training. In production, however, the input training corpus typically consists of a large collection of documents whose combined size can span several terabytes.


In [1]:
with open("data/corpus.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

print(f"Read characters corpus: {len(corpus)}")

Read characters corpus: 20479


## Step 2 (a): Build Vocabulary

Extracting unique words from the corpus.

In [2]:
import re

# Split by words
preprocessed = re.split(r'([,.:;?!_"()\']|--|\s)', corpus)

# Strip whitespace chars
preprocessed = [item.strip() for item in preprocessed if item.strip()]

# Build list of unique words from the corpus
all_words = sorted(set(preprocessed))

# Build vocabulary
vocabulary = { word:integer for integer, word in enumerate(all_words) }

len(vocabulary)

1130

## Step 3: Tokenization

Tokenizer has two operations:
- Encoding: Text to Token ID.
- Decoding: Token ID to Text.


<img src="images/tokenization-ops.png" width="40%" />

### (a): Naive Tokenization

Just split the text into words based on the space character and consider that as token.

<img src="images/tokenization.png" width="40%" />

In [3]:
import re

class SimpleTokenizer:
	"""Maintain a dictionary of tokens to IDs i.e. vocabulary and a dictionary of ID to token."""
	def __init__(self, vocabulary):
		self.token_to_id = vocabulary
		self.id_to_token = { id:token for token, id in vocabulary.items() }

	"""Encode the given text by splitting into tokens and then ecoding them into integer IDs."""
	def encode(self, text):
		## tokenize text
		tokenization_regex = r'([,.:;?_!"()\']|--|\s)'
		tokens = re.split(tokenization_regex, text)

		## strip whitespaces
		tokens = [ token.strip() for token in tokens if token.strip() ]

		ids = [ self.token_to_id[token] for token in tokens ]
		
		return ids

	"""Decode the given integer ids back to the corresponding tokenized text."""
	def decode(self, ids):
		tokens = [ self.id_to_token[id] for id in ids ]
		text = ' '.join(tokens)
		
		## places back the spaces before specified punctuations.
		text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
		
		return text

### Usage of `SimpleTokenizer`

In [4]:
text = "It's the last he painted, you know, Mrs. Gisburn said"

simple_tokenizer = SimpleTokenizer(vocabulary) 

token_ids = simple_tokenizer.encode(text)

print(f"token_ids: {token_ids}")

original_text = simple_tokenizer.decode(token_ids)

print(f"original_text: {original_text}")

failed_case_text = "It's the last he painted, you know, Mrs. Gisburn said Hello, World"
# token_ids = simple_tokenizer.encode(failed_case_text) // Uncomment this like to experience the limitations of `SimpleTokenizer`

token_ids: [56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851]
original_text: It' s the last he painted, you know, Mrs. Gisburn said


### (b): Enhanced Tokenization

**Problem:** Unable to handle unknown words in the corpus.

**Solution:** Extend the vocabulary to handle unknown words and end of text to mark end of docs when corpus spans various docs.

<img src="images/vocab-extension.png" width="40%" />

In [5]:
print(f"Originally vocabulary size based corpus: {len(vocabulary)}")

all_words.extend(["<|endoftext|>", "<|unk|>"])
vocabulary = { token:integer for integer, token in enumerate(all_words) }

print(f"Extended vocabulary size: {len(vocabulary)}")

Originally vocabulary size based corpus: 1130
Extended vocabulary size: 1132


In [6]:
class EnhancedTokenizer:
        """Maintain a dictionary of tokens to IDs i.e. vocabulary and a dictionary of ID to token."""
        def __init__(self, vocabulary):
                self.token_to_id = vocabulary
                self.id_to_token = { id:token for token, id in vocabulary.items() }

        """Encode the given text by splitting into tokens and then ecoding them into integer IDs."""
        def encode(self, text):
                ## tokenize text
                tokenization_regex = r'([,.:;?_!"()\']|--|\s)'
                tokens = re.split(tokenization_regex, text)

                ## strip whitespaces
                tokens = [ token.strip() for token in tokens if token.strip() ]
                
                # fallback to <|unk|> token in case vocabulary does not contain the token. 
                tokens = [ token if token in self.token_to_id else "<|unk|>" for token in tokens ]

                ids = [ self.token_to_id[token] for token in tokens ]

                return ids

        """Decode the given integer ids back to the corresponding tokenized text."""
        def decode(self, ids):
                tokens = [ self.id_to_token[id] for id in ids ]
                text = ' '.join(tokens)

                ## places back the spaces before specified punctuations.
                text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

                return text

### Usage of `EnhancedTokenizer`

In [7]:
text = "It's the last he painted, you know, Mrs. Gisburn said Hello, World"

enhanced_tokenizer = EnhancedTokenizer(vocabulary) 

token_ids = enhanced_tokenizer.encode(text)

print(f"token_ids: {token_ids}")

original_text = enhanced_tokenizer.decode(token_ids)

print(f"original_text: {original_text}")

token_ids: [56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 67, 7, 38, 851, 1131, 5, 1131]
original_text: It' s the last he painted, you know, Mrs. Gisburn said <|unk|>, <|unk|>


**Note:** _Limitation of `SimplerTokenizer` has been fixed and encoded successfully and decoded back <|unk|> token for unknown words like "Hello" and "World"._

### (c): Production-grade `Tokenizer`

**Problem:** Treating every unseen word as the same unknown token prevents the model from distinguishing between different unknown words. Every unseen word is unique because it is composed of a unique sequence of characters. In production, the model will inevitably encounter words, names, or domain-specific terms that were never present in the training data.

**Solution:** Instead of representing every unseen word with a single unknown token, split it into smaller subwords whenever possible using entries already present in the vocabulary. If no matching subwords exist, continue breaking it down until individual characters (or the smallest supported units) can be represented. This allows each unseen word to have a unique token sequence while still using the existing vocabulary.

We will be using an open-source [implementation](https://github.com/openai/tiktoken) of [byte pair encoding algorithm](https://en.wikipedia.org/wiki/Byte-pair_encoding) instead of implementing ourselves.

<img src="images/byte-pair-encoding.png" width="40%" />

**Note:** _This is why, in modern LLMs, tokens are not exactly the same as words. Instead, they are often smaller units, so approx. one token corresponds to about **0.75 words.**_

In [8]:
import tiktoken

class Tokenizer:
    def __init__(self, model_name: str = "gpt2"):
        self.tokenizer = tiktoken.get_encoding(model_name)

    def encode(self, text: str) -> list[int]:
        return self.tokenizer.encode(text)

    def decode(self, token_ids: list[int]) -> str:
        return self.tokenizer.decode(token_ids)

    def __getattr__(self, name):
        # Dynamically forward any missing attributes (like n_vocab) to tiktoken
        return getattr(self.tokenizer, name)


### Usage of `Tokenizer`

In [9]:
text = "It's the last he painted, you know, Mrs. Gisburn said Hello, World"

tokenizer = Tokenizer() 
token_ids = tokenizer.encode(text)

print(f"token_ids: {token_ids}")

original_text = tokenizer.decode(token_ids)

print(f"original_text: {original_text}")

token_ids: [1026, 338, 262, 938, 339, 13055, 11, 345, 760, 11, 9074, 13, 402, 271, 10899, 531, 18435, 11, 2159]
original_text: It's the last he painted, you know, Mrs. Gisburn said Hello, World


## Step 4: Generate Training Samples for Self-Supervised Learning

One of the hallmarks of LLMs is that they predict only **one token at a time**.

This seemingly simple objective is what makes LLMs scale so effectively. 


Since every next token in the training corpus naturally serves as the target label, no manual annotation is required. As a result, LLMs can be trained on massive text corpora using self-supervised learning, avoiding the costly and time-consuming labeling effort required by traditional supervised machine learning systems.

<img src="images/data-sampling.png" width="40%" />

### (a): Essence of the Data Sampling

In [10]:
# Encode the corpus with `Tokenizer`
tokenizer = Tokenizer()
encoded_corpus = tokenizer.encode(corpus)
len(token_ids)

# Self-Supervised Sampling based on Context Window Size
context_window_size = 4

print(f"x -------> y")
for i in range(1, context_window_size+1):
    x = encoded_corpus[:i]
    y = encoded_corpus[i]

    print(f"Input: {tokenizer.decode(x)} ----------> Target: {tokenizer.decode([y])}")

x -------> y
Input: I ----------> Target:  H
Input: I H ----------> Target: AD
Input: I HAD ----------> Target:  always
Input: I HAD always ----------> Target:  thought


<img src="images/data-sampling-2.png" width="40%" />

**Note:** _This shows we are able to use context side as input and desired side as labelled data in the LLM training phase._

### (b): Production-grade Data Sampling 

The essence we explained for the data sampling is the same but we will be going to use pytorch built-in data loader implementations instead of re-inventing the wheel.

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, context_window_size, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt)
        assert len(token_ids) > context_window_size, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - context_window_size, stride):
            input_chunk = token_ids[i:i + context_window_size]
            target_chunk = token_ids[i + 1: i + context_window_size + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, tokenizer, batch_size=4, context_window_size=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, context_window_size, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

### Use the DataLoader

In [12]:
## Sliding window approach to create input and target sequences for training
dataloader = create_dataloader_v1(corpus, tokenizer, batch_size=1, context_window_size=4, stride=1, shuffle=False)
data_iter = iter(dataloader)

first_batch = next(data_iter)
print(first_batch)

second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


**Note:** _Tokens are shifted by 1 position in both batches._

### What does `stride` parameter control?

As we can see there is lot of overlap between batches inputs:
- First batch has inputs: `[40, 367, 2885, 1464]` and targets: `[367, 2885, 1464, 1807]`
- Second batch has inputs: `[367, 2885, 1464, 1807]` and targets: `[2885, 1464, 1807, 3619]`


During LLMs training phase, it will be seeing the same inputs and targets multiple times which leads to overfitting.

Hence we can make the batches non-overlapping using `stride` param.


<img src="images/data-sampling-3.png" width="40%" />


### What does `batch_size` parameter control?
With `batch_size=1`, we have one example where saw tokens are shifted by 1 position to have the targets for the inputs.

Effeciently train LLMs, it is good practice to use larger value like `batch_size=8` where each row in inputs and targets represent one training example.  

In [13]:
## Sliding window approach to create input and target sequences for training
dataloader = create_dataloader_v1(corpus, tokenizer, batch_size=8, context_window_size=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Step 5: Map Token IDs to Vector Representations (Embeddings)

Token IDs are one-to-one discrete representations of tokens.

However, as we discussed in the **Why** section of this computational essay, representing words as discrete IDs is not sufficient for reasoning. We need a way to represent their meanings in a continuous form.

Our training corpus currently has a vocabulary of approximately **50,000** unique tokens.

Before scaling to a vocabulary of this size of corpus, let's first understand the essence of mapping token IDs to vector representations, known as **embeddings**, which capture the meanings of tokens in a continuous space with fewer token IDs.

For simplicity, let's consider the following example:
- Input token IDs: `[2, 3, 5, 1]`
- Vocabulary size: 6 tokens
- Embedding dimension: 3

Our goal is to transform each token ID in the input sequence into its corresponding 3-dimensional embedding vector. In other words, we want to replace each discrete token ID with a continuous vector representation that captures the meaning associated with that token.


In [14]:
inputs = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

### Initialize the pytorch embedding layer 

It has weight matrix which gets updated during training phase to learn the meaning of the words being used in the corpus in the sense by learning the pattern of their usage.

In [15]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
embedding_layer.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

**Note:** _Unlike `torch.nn.Linear` layer the `torch.nn.Embedding` layer provides fast quick look up inside its weight matrix._

In [16]:
print(f"First row of weight matrix: {embedding_layer(torch.tensor([0]))}")
print(f"Third row of weight matrix: {embedding_layer(torch.tensor([2]))}")
print(f"Last row of weight matrix: {embedding_layer(torch.tensor([5]))}")

First row of weight matrix: tensor([[ 0.3374, -0.1778, -0.1690]], grad_fn=<EmbeddingBackward0>)
Third row of weight matrix: tensor([[ 1.2753, -0.2010, -0.1606]], grad_fn=<EmbeddingBackward0>)
Last row of weight matrix: tensor([[-2.8400, -0.7849, -1.4096]], grad_fn=<EmbeddingBackward0>)


We can lookup rows of `embedding_layer.weight` using inputs token IDs like:

In [17]:
token_embeddings = embedding_layer(inputs)
token_embeddings

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

**Note:** _Last result returns third, fourth, sixth and second rows of embedding_layer.weight matrix._

## Step 6: Make the Embedding Position Aware

Initialize another `torch.nn.Embedding` of context window size.

Then add the pos_embeddings to the previous embeddings layer to make up positional aware input embeddings layer.

In [18]:
context_window_size = 4

## Initialize another Embedding layer for position information
pos_embedding_layer = torch.nn.Embedding(context_window_size, output_dim)

## Get the weight matrix values based on the position using `torch.arange(context_window_size) = [0, 1, 2, 3]`
pos_embeddings = pos_embedding_layer(torch.arange(context_window_size))

pos_embeddings

tensor([[-0.6307,  1.2340,  0.3127],
        [ 0.6972, -0.9950, -1.1476],
        [-0.9178,  0.9045, -2.0975],
        [ 1.1558, -1.2157,  0.1295]], grad_fn=<EmbeddingBackward0>)

### Now, incorporate this position weights to token embedding layer.

<img src="images/positional-embeddings.png" width="40%" />

In [19]:
input_embeddings = pos_embeddings + token_embeddings
input_embeddings

tensor([[ 0.6446,  1.0331,  0.1521],
        [ 0.2957, -0.0285, -2.2958],
        [-3.7578,  0.1197, -3.5071],
        [ 2.0735,  0.3653,  1.4306]], grad_fn=<AddBackward0>)

## Step 7: Stitching Together 

In [20]:
# Select important parameters
embedding_dim = 256
context_window_size = 4

# Load corpus for training
with open("data/corpus.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

# Initialize BPE tokenizer
tokenizer = Tokenizer()

# Sliding window approach to sample input and target sequences for training
dataloader = create_dataloader_v1(corpus, tokenizer, batch_size=8, context_window_size=context_window_size, stride=context_window_size, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

# Embedding Layer for all tokens in the vocabulary and each tokenID represented in embedding_dim
token_embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, embedding_dim)
token_embeddings = token_embedding_layer(inputs)

# Embedding Layer for all tokens position in the context window and each tokenID represented in embedding_dim
pos_embedding_layer = torch.nn.Embedding(context_window_size, embedding_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_window_size))

# Make the token embeddings aware of positional information
input_embeddings = pos_embeddings + token_embeddings

print(input_embeddings.shape)

torch.Size([8, 4, 256])


**Note:** _The `input_embeddings.shape = [8, 4, 256]` will be fed into the attention attention mechanism which we have already prepared in the previous computational essays._

## What Did We Learn?

In this computational essay, we built the complete data preparation pipeline that transforms raw text into a representation that a GPT model can understand and learn from.

We learned that:

* Words themselves are not knowledge; they are representations of abstract meanings.
* Discrete token IDs are sufficient for storing and indexing text, but they are not sufficient for reasoning.
* To enable reasoning, each token must be mapped to a continuous vector representation, known as an **embedding**, which captures its field of meaning.
* Before words can be represented as embeddings, raw text must first be converted into **tokens** through tokenization. Modern LLMs achieve this by splitting unseen words into meaningful subwords rather than treating every unknown word identically.
* LLMs learn through **self-supervised learning**, where every next token in the corpus naturally serves as the target to be predicted, eliminating the need for manually labeled data.
* Since language is sequential, token embeddings alone are not enough. They must also be enriched with **positional information** so that the model can distinguish not only *what* a token is, but also *where* it appears in the sequence.
* The limitations of modeling the field of meanings using vectors in computer systems is that we can only model the field of meanings in which humans have used words so far, whereas humans can continuously enrich existing meanings and create entirely new ones.

At this point, we have completed the entire data preparation pipeline. Starting from raw textual data, we transformed it into a sequence of contextual inputs that are now ready to be fed into the GPT model.

In the next computational essay, we will finally connect this prepared data with the multi-head causal masked attention mechanism that we developed in the previous essays and see how the GPT model begins to learn the statistical patterns of human language.

<img src="images/embedding-pipeline-overview.png" width="50%" />
